In [9]:
# Backtest

import pandas as pd

df = pd.read_csv("./data/AAPL.csv")

# print(df.isna().sum()) #before it was one null per column
df.dropna(inplace=True)
# print(df.isna().sum()) # after is 0

df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)
print(df.head().describe())


daily_future_return = (df["Adj Close"].shift(-1) - df["Adj Close"]) / df["Adj Close"]
daily_future_return.name = "daily_future_returns"
print(daily_future_return)

           Open      High       Low     Close  Adj Close        Volume
count  5.000000  5.000000  5.000000  5.000000   5.000000  5.000000e+00
mean   0.119643  0.119978  0.119420  0.119420   0.094042  1.821075e+08
std    0.005927  0.006023  0.006023  0.006023   0.004743  1.651951e+08
min    0.113281  0.113281  0.112723  0.112723   0.088768  7.344960e+07
25%    0.115513  0.116071  0.115513  0.115513   0.090965  8.644160e+07
50%    0.118862  0.119420  0.118862  0.118862   0.093603  1.057280e+08
75%    0.122210  0.122210  0.121652  0.121652   0.095800  1.758848e+08
max    0.128348  0.128906  0.128348  0.128348   0.101073  4.690336e+08
Date
1980-12-12   -0.052170
1980-12-15   -0.073403
1980-12-16    0.024750
1980-12-17    0.029000
1980-12-18    0.061024
                ...   
2021-01-25    0.001679
2021-01-26   -0.007684
2021-01-27   -0.034985
2021-01-28   -0.037421
2021-01-29         NaN
Name: daily_future_returns, Length: 10118, dtype: float64


In [10]:
# Create a Series that contains a random boolean array with p=0.5

import numpy as np

np.random.seed(2712)

signal = np.random.randint(0, 2, len(df.index))

long_only_signal = pd.Series(signal, index=df.index, name="long_only_signal")
print(long_only_signal.head())

Date
1980-12-12    1
1980-12-15    0
1980-12-16    1
1980-12-17    1
1980-12-18    1
Name: long_only_signal, dtype: int64


In [11]:
# Backtest the signal

future_return = (df["Adj Close"].shift(-1) - df["Adj Close"]) / df["Adj Close"]

Pnl = long_only_signal * future_return
Pnl.name = "Pnl"
Pnl


Date
1980-12-12   -0.052170
1980-12-15   -0.000000
1980-12-16    0.024750
1980-12-17    0.029000
1980-12-18    0.061024
                ...   
2021-01-25    0.001679
2021-01-26   -0.007684
2021-01-27   -0.034985
2021-01-28   -0.000000
2021-01-29         NaN
Name: Pnl, Length: 10118, dtype: float64

In [12]:
total_invested = long_only_signal.sum()
total_earned = Pnl.sum()


total_return = total_earned / total_invested

print("Total invested:", total_invested)
print("Total earned:", total_earned)
print("Total strategy return:", total_return)

Total invested: 5022
Total earned: 4.562457958401091
Total strategy return: 0.0009084942171248688


In [26]:
import plotly.graph_objects as go


# always buy = signal is always 1
always_buy_signal = pd.Series(1, index=df.index, name="always_buy")
always_buy_pnl = always_buy_signal * future_return
always_buy_pnl.name = "always_buy_pnl"


fig = go.Figure()

fig.add_trace(go.Scatter(x=Pnl.index, y=Pnl, mode="lines",line=dict(color="blue"), name="Random signal PnL"))
fig.add_trace(go.Scatter(x=always_buy_pnl.index, y=always_buy_pnl, mode="lines", name="Always buy PnL"))

fig.update_layout(
    title="Strategy Comparison",
    xaxis_title="Date",
    yaxis_title="PnL",
)

fig.show()